# Base04 — Enriquecimento detalhado das glebas do SICOR Cerrado (2021–2025)

Este notebook acrescenta os dados operacionais e financeiros do SICOR à base territorial completa, **sem eliminar registros territoriais, `REF_BACEN` repetidos ou geometrias repetidas**.

## Unidade preservada

O produto final mantém:

```text
1 linha de entrada da base territorial = 1 linha no resultado
```

A base territorial é vinculada às operações pela chave:

```text
REF_BACEN + NU_ORDEM
```

O `NU_INDICE`, a geometria, o ano-base e o `ID_GLEBA` continuam individualizados. Se o CSV territorial ainda não tiver `ID_GLEBA`, o notebook cria uma numeração sequencial sem excluir nenhuma linha.

## Regra importante

As linhas operacionais são consolidadas somente por `REF_BACEN + NU_ORDEM` antes da junção. Essa consolidação técnica impede que duplicidades administrativas nas bases operacionais multipliquem artificialmente as linhas territoriais. **Não há consolidação final por `REF_BACEN`.**

O notebook não utiliza uma lista auxiliar de referências únicas. O universo de `REF_BACEN` é obtido diretamente do arquivo territorial completo.


In [3]:
# ============================================================
# 1. MONTAR O GOOGLE DRIVE E IMPORTAR BIBLIOTECAS
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import gc
import re
import unicodedata

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.expand_frame_repr", False)

print("Ambiente preparado.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ambiente preparado.


In [4]:
# ============================================================
# 2. CAMINHOS DOS ARQUIVOS
# ============================================================

PASTA_BASE = Path(
    "/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Base"
)

PASTA_ARQUIVOS_GERADOS = Path(
    "/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/"
    "Arquivos gerados e relatorios"
)

ARQUIVO_CERRADO_CONSOLIDADO = (
    PASTA_ARQUIVOS_GERADOS
    / "sicor_cerrado_2021_2025.csv"
)

ANOS = [2021, 2022, 2023, 2024, 2025]


def localizar_arquivo_na_base(nome_arquivo):
    """Localiza um arquivo pelo nome sem escolher silenciosamente entre duplicatas."""
    caminho_direto = PASTA_BASE / nome_arquivo

    if caminho_direto.exists():
        return caminho_direto

    correspondencias = [
        caminho
        for caminho in PASTA_BASE.rglob("*")
        if caminho.is_file()
        and caminho.name.casefold() == nome_arquivo.casefold()
    ]

    if not correspondencias:
        raise FileNotFoundError(
            f"Arquivo não encontrado dentro de {PASTA_BASE}: {nome_arquivo}"
        )

    if len(correspondencias) > 1:
        raise RuntimeError(
            f"Foram encontrados vários arquivos chamados {nome_arquivo}:\n"
            + "\n".join(str(caminho) for caminho in correspondencias)
        )

    return correspondencias[0]


ARQUIVOS_OPERACOES = {
    ano: localizar_arquivo_na_base(
        f"SICOR_OPERACAO_BASICA_ESTADO_{ano}.csv"
    )
    for ano in ANOS
}

ARQUIVO_PROGRAMA = localizar_arquivo_na_base("Programa.csv")
ARQUIVO_CATEGORIA = localizar_arquivo_na_base("CategoriaEmitente.csv")
ARQUIVO_SUBPROGRAMAS = localizar_arquivo_na_base("Subprogramas.csv")

PASTA_SAIDA = (
    PASTA_ARQUIVOS_GERADOS
    / "Base04_enriquecimento_detalhado"
)
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

ARQUIVO_SAIDA = (
    PASTA_SAIDA
    / "sicor_cerrado_2021_2025_enriquecido_detalhado.csv"
)

ARQUIVO_RELATORIO = (
    PASTA_SAIDA
    / "relatorio_base04_enriquecimento_detalhado.csv"
)

ARQUIVO_NAO_VINCULADOS = (
    PASTA_SAIDA
    / "registros_territoriais_nao_vinculados_operacoes.csv"
)

ARQUIVO_AUDITORIA_ORDENS = (
    PASTA_SAIDA
    / "auditoria_consolidacao_operacoes_por_ordem.csv"
)

ARQUIVO_DICIONARIO = (
    PASTA_SAIDA
    / "dicionario_base04_enriquecimento_detalhado.csv"
)

SEPARADOR_SAIDA = ";"
TAMANHO_BLOCO = 100_000

arquivos_obrigatorios = [
    ARQUIVO_CERRADO_CONSOLIDADO,
    ARQUIVO_PROGRAMA,
    ARQUIVO_CATEGORIA,
    ARQUIVO_SUBPROGRAMAS,
    *ARQUIVOS_OPERACOES.values(),
]

arquivos_ausentes = [
    caminho
    for caminho in arquivos_obrigatorios
    if not caminho.exists()
]

for caminho in arquivos_obrigatorios:
    print(
        f"{'OK' if caminho.exists() else 'NÃO ENCONTRADO'} | {caminho}"
    )

if arquivos_ausentes:
    raise FileNotFoundError(
        "Arquivos obrigatórios não encontrados:\n"
        + "\n".join(str(caminho) for caminho in arquivos_ausentes)
    )

print("\nTodos os arquivos obrigatórios foram encontrados.")
print("Pasta de saída:", PASTA_SAIDA)


OK | /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/sicor_cerrado_2021_2025.csv
OK | /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Base/Programa.csv
OK | /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Base/CategoriaEmitente.csv
OK | /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Base/Subprogramas.csv
OK | /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Base/SICOR_OPERACAO_BASICA_ESTADO_2021.csv
OK | /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Base/SICOR_OPERACAO_BASICA_ESTADO_2022.csv
OK | /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Base/SICOR_OPERACAO_BASICA_ESTADO_2023.csv
OK | /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Base/SICOR_OPERACAO_BASICA_ESTADO_2024.csv
OK | /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Base/SICOR_OPERACAO_BASICA_ESTADO_2025.csv

Todos os arquivos obrigatórios foram encontrados.
Pasta de saída: /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/Base04_e

In [5]:
# ============================================================
# 3. FUNÇÕES DE LEITURA E NORMALIZAÇÃO
# ============================================================

CODIFICACOES = ["utf-8-sig", "utf-8", "latin-1", "cp1252"]
SEPARADORES = [";", ",", "\t", "|"]


def remover_acentos(texto):
    return "".join(
        caractere
        for caractere in unicodedata.normalize("NFKD", str(texto))
        if not unicodedata.combining(caractere)
    )


def normalizar_nome_coluna(nome):
    nome = remover_acentos(nome).upper().strip()
    nome = nome.replace("#", "")
    nome = re.sub(r"[^A-Z0-9]+", "_", nome)
    nome = re.sub(r"_+", "_", nome).strip("_")
    return nome


def normalizar_colunas(df):
    df = df.copy()
    novas_colunas = [
        normalizar_nome_coluna(coluna)
        for coluna in df.columns
    ]

    if len(novas_colunas) != len(set(novas_colunas)):
        duplicadas = pd.Series(novas_colunas)
        duplicadas = duplicadas[duplicadas.duplicated(keep=False)].tolist()
        raise ValueError(
            "A normalização produziu nomes de colunas duplicados: "
            f"{sorted(set(duplicadas))}"
        )

    df.columns = novas_colunas
    return df


def detectar_leitura(caminho, aceitar_uma_coluna=True):
    ultimo_erro = None

    for encoding in CODIFICACOES:
        for separador in SEPARADORES:
            try:
                teste = pd.read_csv(
                    caminho,
                    sep=separador,
                    encoding=encoding,
                    nrows=5,
                    low_memory=False,
                )

                if teste.shape[1] < 1:
                    continue

                if not aceitar_uma_coluna and teste.shape[1] == 1:
                    continue

                teste = normalizar_colunas(teste)
                return encoding, separador, list(teste.columns)

            except Exception as erro:
                ultimo_erro = erro

    raise RuntimeError(
        f"Não foi possível ler {caminho.name}. Último erro: {ultimo_erro}"
    )


def ler_csv_robusto(caminho, nome, aceitar_uma_coluna=True):
    encoding, separador, _ = detectar_leitura(
        caminho,
        aceitar_uma_coluna=aceitar_uma_coluna,
    )

    df = pd.read_csv(
        caminho,
        sep=separador,
        encoding=encoding,
        dtype=str,
        low_memory=False,
    )

    df = normalizar_colunas(df)

    print(
        f"{nome}: {len(df):,} linhas × {len(df.columns):,} colunas | "
        f"encoding={encoding} | separador={repr(separador)}"
    )

    return df


def localizar_coluna(colunas, candidatos, obrigatoria=True):
    for candidato in candidatos:
        candidato_normalizado = normalizar_nome_coluna(candidato)
        if candidato_normalizado in colunas:
            return candidato_normalizado

    if obrigatoria:
        raise KeyError(
            "Nenhuma das colunas esperadas foi encontrada: "
            f"{candidatos}. Colunas disponíveis: {list(colunas)}"
        )

    return None


def padronizar_codigo(serie):
    return (
        serie.astype("string")
        .str.strip()
        .str.lstrip("\ufeff")
        .str.replace(r"\.0$", "", regex=True)
        .replace(
            {
                "": pd.NA,
                "NAN": pd.NA,
                "NONE": pd.NA,
                "<NA>": pd.NA,
            }
        )
    )


def numero_brasileiro(serie):
    texto = serie.astype("string").str.strip()
    mascara_virgula = texto.str.contains(",", na=False)

    texto = texto.where(
        ~mascara_virgula,
        texto.str.replace(".", "", regex=False)
             .str.replace(",", ".", regex=False),
    )

    return pd.to_numeric(texto, errors="coerce")


def primeiro_nao_nulo(serie):
    valores = serie.dropna()

    if pd.api.types.is_string_dtype(valores.dtype):
        valores = valores[valores.astype(str).str.strip().ne("")]

    return valores.iloc[0] if len(valores) else pd.NA


def texto_sem_acento_maiusculo(serie):
    return (
        serie.astype("string")
        .fillna("")
        .map(remover_acentos)
        .str.upper()
        .str.strip()
    )


In [6]:
# ============================================================
# 4. LEITURA INTEGRAL DA BASE TERRITORIAL DA BASE02
# ============================================================

encoding_cerrado, separador_cerrado, colunas_cerrado = detectar_leitura(
    ARQUIVO_CERRADO_CONSOLIDADO,
    aceitar_uma_coluna=False,
)

COLUNA_REF_CERRADO = localizar_coluna(
    colunas_cerrado,
    ["REF_BACEN", "#REF_BACEN", "CD_REF_BACEN"],
)

COLUNA_ORDEM_CERRADO = localizar_coluna(
    colunas_cerrado,
    ["NU_ORDEM", "NU_ORDEM_GLEBA", "NUM_ORDEM", "ORDEM"],
)

COLUNA_INDICE_CERRADO = localizar_coluna(
    colunas_cerrado,
    ["NU_INDICE", "NUM_INDICE", "INDICE"],
)

COLUNA_ANO_CERRADO = localizar_coluna(
    colunas_cerrado,
    ["ANO_BASE", "ANO"],
)

COLUNA_GEOMETRIA = localizar_coluna(
    colunas_cerrado,
    ["GT_GEOMETRIA", "GEOMETRIA", "GEOMETRY", "WKT"],
)

TEM_ID_GLEBA = "ID_GLEBA" in colunas_cerrado

renomeacoes_base = {
    COLUNA_REF_CERRADO: "REF_BACEN",
    COLUNA_ORDEM_CERRADO: "NU_ORDEM_GLEBA",
    COLUNA_INDICE_CERRADO: "NU_INDICE",
    COLUNA_ANO_CERRADO: "ANO_BASE",
    COLUNA_GEOMETRIA: "GT_GEOMETRIA",
}

partes_glebas = []
proximo_id = 1
ordem_interna = 0

leitor_glebas = pd.read_csv(
    ARQUIVO_CERRADO_CONSOLIDADO,
    sep=separador_cerrado,
    encoding=encoding_cerrado,
    dtype=str,
    chunksize=TAMANHO_BLOCO,
    low_memory=False,
)

for numero_bloco, bloco in enumerate(leitor_glebas, start=1):
    bloco = normalizar_colunas(bloco)
    bloco = bloco.rename(columns=renomeacoes_base)

    quantidade_bloco = len(bloco)

    if not TEM_ID_GLEBA:
        bloco.insert(
            0,
            "ID_GLEBA",
            range(proximo_id, proximo_id + quantidade_bloco),
        )
        proximo_id += quantidade_bloco
    else:
        bloco["ID_GLEBA"] = padronizar_codigo(bloco["ID_GLEBA"])

    bloco["REF_BACEN"] = padronizar_codigo(bloco["REF_BACEN"])
    bloco["NU_ORDEM_GLEBA"] = padronizar_codigo(
        bloco["NU_ORDEM_GLEBA"]
    )
    bloco["NU_INDICE"] = padronizar_codigo(bloco["NU_INDICE"])

    bloco["_ORDEM_LINHA_BASE04"] = range(
        ordem_interna,
        ordem_interna + quantidade_bloco,
    )
    ordem_interna += quantidade_bloco

    partes_glebas.append(bloco)

    print(
        f"Bloco {numero_bloco:03d}: {quantidade_bloco:,} linhas | "
        f"acumulado={ordem_interna:,}"
    )

    del bloco
    gc.collect()

glebas = pd.concat(
    partes_glebas,
    ignore_index=True,
    sort=False,
)

del partes_glebas
gc.collect()

colunas_originais_base = [
    coluna
    for coluna in glebas.columns
    if coluna != "_ORDEM_LINHA_BASE04"
]

quantidade_registros_origem = len(glebas)
quantidade_refs_origem = glebas["REF_BACEN"].nunique(dropna=True)
conjunto_refs = set(glebas["REF_BACEN"].dropna())

if glebas["ID_GLEBA"].isna().any():
    raise RuntimeError("Existem valores nulos em ID_GLEBA.")

if not glebas["ID_GLEBA"].is_unique:
    raise RuntimeError("ID_GLEBA não é único na base territorial.")

if glebas["GT_GEOMETRIA"].isna().any():
    raise RuntimeError("Existem geometrias vazias na base territorial.")

print("\nBASE TERRITORIAL CARREGADA")
print("=" * 80)
print("Registros territoriais:", f"{quantidade_registros_origem:,}")
print("REF_BACEN distintos:", f"{quantidade_refs_origem:,}")
print("ID_GLEBA já existia no arquivo:", "Sim" if TEM_ID_GLEBA else "Não")
print("ID_GLEBA únicos:", f"{glebas['ID_GLEBA'].nunique():,}")
print("Geometrias WKT distintas:", f"{glebas['GT_GEOMETRIA'].nunique():,}")

display(glebas.head())


Bloco 001: 100,000 linhas | acumulado=100,000
Bloco 002: 100,000 linhas | acumulado=200,000
Bloco 003: 100,000 linhas | acumulado=300,000
Bloco 004: 100,000 linhas | acumulado=400,000
Bloco 005: 80,679 linhas | acumulado=480,679

BASE TERRITORIAL CARREGADA
Registros territoriais: 480,679
REF_BACEN distintos: 334,166
ID_GLEBA já existia no arquivo: Sim
ID_GLEBA únicos: 480,679
Geometrias WKT distintas: 395,575


,ID_GLEBA,REF_BACEN,NU_ORDEM_GLEBA,NU_INDICE,GT_GEOMETRIA,ANO_BASE,_ORDEM_LINHA_BASE04
0,1,511405404,1,0,"POLYGON ((-47.521674 -18.527653 0,-47.521697 -18.528289 0,-47.519128 -18.528105 0,-47.519506 -18.526262 0,-47.519955 -18.526712 0,-47.520342 -18.527134 0,-47.520581 -18.527374 ...",2021,0
1,2,511405405,1,0,"POLYGON ((-47.579356 -18.906067 0,-47.578942 -18.906589 0,-47.578871 -18.907144 0,-47.578482 -18.907021 0,-47.57761 -18.906084 0,-47.577963 -18.905703 0,-47.57713 -18.905449999...",2021,1
2,3,511405424,1,0,"POLYGON ((-44.441404 -16.832015 0,-44.441419 -16.831524 0,-44.440824 -16.831596 0,-44.439808999999897 -16.831699 0,-44.43956 -16.830835 0,-44.439273 -16.830788 0,-44.43887 -16....",2021,2
3,4,511405446,1,0,"POLYGON ((-44.965694 -20.079657 0,-44.967058 -20.077494 0,-44.968458 -20.078419 0,-44.970597 -20.078106 0,-44.972607 -20.077254 0,-44.972477 -20.076066 0,-44.972887 -20.074928 ...",2021,3
4,5,511405448,1,0,"POLYGON ((-45.075154 -18.327891 0,-45.077628 -18.329547 0,-45.078869 -18.329854 0,-45.079659 -18.329899 0,-45.08048 -18.330235 0,-45.081125 -18.331326 0,-45.081536 -18.331539 0...",2021,4


In [7]:
# ============================================================
# 5. LEITURA E PREPARAÇÃO DAS TABELAS DE DOMÍNIO
# ============================================================

programa = ler_csv_robusto(
    ARQUIVO_PROGRAMA,
    "Programa",
    aceitar_uma_coluna=False,
)

categoria = ler_csv_robusto(
    ARQUIVO_CATEGORIA,
    "Categoria do emitente",
    aceitar_uma_coluna=False,
)

subprogramas = ler_csv_robusto(
    ARQUIVO_SUBPROGRAMAS,
    "Subprogramas",
    aceitar_uma_coluna=False,
)


def localizar_coluna_flexivel(
    df,
    candidatos_exatos,
    termos_entidade,
    tipo,
    obrigatoria=True,
    colunas_excluidas=None,
):
    colunas_excluidas = set(colunas_excluidas or [])
    colunas = [
        coluna
        for coluna in df.columns
        if coluna not in colunas_excluidas
    ]

    for candidato in candidatos_exatos:
        candidato = normalizar_nome_coluna(candidato)
        if candidato in colunas:
            return candidato

    termos_entidade = [
        normalizar_nome_coluna(termo)
        for termo in termos_entidade
    ]

    termos_tipo = (
        ["CD", "COD", "CODIGO", "ID"]
        if tipo == "codigo"
        else ["DESC", "DESCR", "DESCRICAO", "NOME"]
    )

    for coluna in colunas:
        possui_entidade = any(
            termo in coluna
            for termo in termos_entidade
        )
        possui_tipo = any(
            termo in coluna
            for termo in termos_tipo
        )

        if possui_entidade and possui_tipo:
            return coluna

    nomes_genericos = (
        ["CODIGO", "COD", "CD", "ID"]
        if tipo == "codigo"
        else ["DESCRICAO", "DESC", "NOME"]
    )

    for nome in nomes_genericos:
        if nome in colunas:
            return nome

    if obrigatoria:
        raise KeyError(
            f"Não foi possível localizar uma coluna de {tipo}. "
            f"Colunas disponíveis: {colunas}"
        )

    return None


def preparar_dimensao(
    df,
    nome_tabela,
    candidatos_codigo,
    nome_codigo,
    nome_descricao,
    termos_entidade,
    candidatos_codigo_secundario=None,
    nome_codigo_secundario=None,
    termos_entidade_secundaria=None,
):
    df = df.copy()

    coluna_codigo = localizar_coluna_flexivel(
        df,
        candidatos_codigo,
        termos_entidade,
        tipo="codigo",
    )

    df = df.rename(columns={coluna_codigo: nome_codigo})
    df[nome_codigo] = padronizar_codigo(df[nome_codigo])

    colunas_excluidas = {nome_codigo}
    codigo_secundario_confirmado = None

    if candidatos_codigo_secundario:
        coluna_secundaria = localizar_coluna_flexivel(
            df,
            candidatos_codigo_secundario,
            termos_entidade_secundaria or [],
            tipo="codigo",
            obrigatoria=False,
            colunas_excluidas=colunas_excluidas,
        )

        if coluna_secundaria:
            df = df.rename(
                columns={coluna_secundaria: nome_codigo_secundario}
            )
            df[nome_codigo_secundario] = padronizar_codigo(
                df[nome_codigo_secundario]
            )
            colunas_excluidas.add(nome_codigo_secundario)
            codigo_secundario_confirmado = nome_codigo_secundario

    coluna_descricao = localizar_coluna_flexivel(
        df,
        [nome_descricao, "DESCRICAO", "DESC", "NOME"],
        termos_entidade,
        tipo="descricao",
        obrigatoria=False,
        colunas_excluidas=colunas_excluidas,
    )

    if coluna_descricao:
        df = df.rename(columns={coluna_descricao: nome_descricao})
    else:
        df[nome_descricao] = pd.NA

    chave = (
        [codigo_secundario_confirmado, nome_codigo]
        if codigo_secundario_confirmado
        else [nome_codigo]
    )

    dimensao = df[chave + [nome_descricao]].copy()
    dimensao[nome_descricao] = (
        dimensao[nome_descricao].astype("string").str.strip()
    )
    dimensao = dimensao.dropna(subset=[nome_codigo])

    dimensao = (
        dimensao.groupby(chave, dropna=False, as_index=False)
        .agg({nome_descricao: primeiro_nao_nulo})
    )

    if dimensao.duplicated(subset=chave).any():
        raise RuntimeError(
            f"A dimensão {nome_tabela} permaneceu com chaves duplicadas."
        )

    print(
        f"{nome_tabela}: {len(dimensao):,} chaves preparadas | "
        f"chave={chave}"
    )

    return dimensao


programa_dim = preparar_dimensao(
    programa,
    "Programa",
    ["CD_PROGRAMA", "COD_PROGRAMA", "CODIGO_PROGRAMA", "CODIGO"],
    "CD_PROGRAMA",
    "DESC_PROGRAMA",
    ["PROGRAMA"],
)

categoria_dim = preparar_dimensao(
    categoria,
    "Categoria do emitente",
    [
        "CD_CATEG_EMITENTE",
        "COD_CATEG_EMITENTE",
        "CODIGO_CATEGORIA_EMITENTE",
        "CD_CATEGORIA",
        "CODIGO",
    ],
    "CD_CATEG_EMITENTE",
    "DESC_CATEG_EMITENTE",
    ["CATEGORIA", "CATEG", "EMITENTE"],
)

subprograma_dim = preparar_dimensao(
    subprogramas,
    "Subprogramas",
    ["CD_SUBPROGRAMA", "COD_SUBPROGRAMA", "CODIGO_SUBPROGRAMA", "CODIGO"],
    "CD_SUBPROGRAMA",
    "DESC_SUBPROGRAMA",
    ["SUBPROGRAMA"],
    candidatos_codigo_secundario=[
        "CD_PROGRAMA",
        "COD_PROGRAMA",
        "CODIGO_PROGRAMA",
    ],
    nome_codigo_secundario="CD_PROGRAMA",
    termos_entidade_secundaria=["PROGRAMA"],
)

display(programa_dim.head())
display(categoria_dim.head())
display(subprograma_dim.head())


Programa: 40 linhas × 5 colunas | encoding=latin-1 | separador=';'
Categoria do emitente: 6 linhas × 6 colunas | encoding=latin-1 | separador=';'
Subprogramas: 85 linhas × 4 colunas | encoding=latin-1 | separador=';'
Programa: 40 chaves preparadas | chave=['CD_PROGRAMA']
Categoria do emitente: 6 chaves preparadas | chave=['CD_CATEG_EMITENTE']
Subprogramas: 85 chaves preparadas | chave=['CD_PROGRAMA', 'CD_SUBPROGRAMA']


,CD_PROGRAMA,DESC_PROGRAMA
0,0001,PRONAF - PROGRAMA NACIONAL DE FORTALECIMENTO DA AGRICULTURA FAMILIAR
1,0050,PRONAMP - PROGRAMA NACIONAL DE APOIO AO MÉDIO PRODUTOR RURAL
2,0070,FUNCAFÉ (PROGRAMA DE DEFESA DA ECONOMIA CAFEEIRA)
3,0100,PRLC-BA (PROG RECUP LAVOURA CACAUEIRA BAIANA) ENCERRADO
4,0110,PRODECER III - PROG COOP NIPO-BRASILEIRA P DESENV DOS CERRADOS - ENCERRADO


,CD_CATEG_EMITENTE,DESC_CATEG_EMITENTE
0,2222,Pequeno Produtor Rural
1,3333,Médio Produtor Rural
2,4444,Grande Produtor Rural
3,5555,Todas Categorias de Produtores Rurais
4,7777,Repasse a Cooperativa Singular de Crédito


,CD_PROGRAMA,CD_SUBPROGRAMA,DESC_SUBPROGRAMA
0,0001,0001,Custeio (MCR 10-4)
1,0001,0002,Mais Alimentos (MCR 10-5)
2,0001,0003,Agroindústria (investimento) (MCR 10-6)
3,0001,0004,Crédito Invest Sist Agroflorestais (Pronaf Floresta) MCR10-7
4,0001,0005,Créd Invest Conviv Semiárido (Pronaf Semiárido) MCR 10-8


In [8]:
# ============================================================
# 6. LEITURA DAS OPERAÇÕES DE 2021 A 2025
# ============================================================

ALIASES_OPERACOES = {
    "REF_BACEN": ["REF_BACEN", "#REF_BACEN", "CD_REF_BACEN"],
    "NU_ORDEM": ["NU_ORDEM", "NUM_ORDEM", "ORDEM"],
    "DT_EMISSAO": ["DT_EMISSAO", "DATA_EMISSAO", "DATA_DA_EMISSAO"],
    "CD_ESTADO": ["CD_ESTADO", "COD_ESTADO", "CODIGO_ESTADO"],
    "SG_UF": ["SG_UF", "UF", "SIGLA_UF"],
    "CD_PROGRAMA": ["CD_PROGRAMA", "COD_PROGRAMA", "CODIGO_PROGRAMA"],
    "CD_CATEG_EMITENTE": [
        "CD_CATEG_EMITENTE",
        "COD_CATEG_EMITENTE",
        "CODIGO_CATEGORIA_EMITENTE",
    ],
    "CD_SUBPROGRAMA": [
        "CD_SUBPROGRAMA",
        "COD_SUBPROGRAMA",
        "CODIGO_SUBPROGRAMA",
    ],
    "VL_PARC_CREDITO": [
        "VL_PARC_CREDITO",
        "VALOR_PARCELA_CREDITO",
        "VALOR_CREDITO",
    ],
    "VL_AREA_INFORMADA": ["VL_AREA_INFORMADA", "AREA_INFORMADA"],
    "VL_AREA_FINANC": [
        "VL_AREA_FINANC",
        "VL_AREA_FINANCIADA",
        "AREA_FINANCIADA",
    ],
}

partes_operacoes = []
totais_operacoes_ano = {}

for ano, caminho in ARQUIVOS_OPERACOES.items():
    encoding, separador, colunas_arquivo = detectar_leitura(
        caminho,
        aceitar_uma_coluna=False,
    )

    mapeamento = {}

    for nome_padrao, candidatos in ALIASES_OPERACOES.items():
        coluna_encontrada = localizar_coluna(
            colunas_arquivo,
            candidatos,
            obrigatoria=nome_padrao in {"REF_BACEN", "NU_ORDEM"},
        )

        if coluna_encontrada:
            mapeamento[coluna_encontrada] = nome_padrao

    colunas_normalizadas_desejadas = set(mapeamento.keys())
    total_ano = 0

    print("\n" + "=" * 80)
    print(f"PROCESSANDO OPERAÇÕES DE {ano}")
    print("=" * 80)

    leitor_operacoes = pd.read_csv(
        caminho,
        sep=separador,
        encoding=encoding,
        usecols=lambda nome_original: (
            normalizar_nome_coluna(nome_original)
            in colunas_normalizadas_desejadas
        ),
        dtype=str,
        chunksize=TAMANHO_BLOCO,
        low_memory=False,
    )

    for numero_bloco, bloco in enumerate(leitor_operacoes, start=1):
        bloco = normalizar_colunas(bloco)
        bloco = bloco.rename(columns=mapeamento)

        bloco["REF_BACEN"] = padronizar_codigo(bloco["REF_BACEN"])
        bloco["NU_ORDEM"] = padronizar_codigo(bloco["NU_ORDEM"])
        bloco = bloco.dropna(subset=["REF_BACEN"])

        # O conjunto vem diretamente da base territorial completa.
        bloco = bloco[bloco["REF_BACEN"].isin(conjunto_refs)].copy()

        if bloco.empty:
            del bloco
            gc.collect()
            continue

        bloco["ANO_ARQUIVO_OPERACAO"] = ano
        partes_operacoes.append(bloco)
        total_ano += len(bloco)

        print(
            f"Bloco {numero_bloco:03d}: {len(bloco):,} linhas | "
            f"acumulado do ano={total_ano:,}"
        )

        del bloco
        gc.collect()

    totais_operacoes_ano[ano] = total_ano
    print(f"Total selecionado em {ano}: {total_ano:,}")

if not partes_operacoes:
    raise RuntimeError(
        "Nenhuma operação foi localizada para os REF_BACEN da base territorial."
    )

operacoes = pd.concat(
    partes_operacoes,
    ignore_index=True,
    sort=False,
)

del partes_operacoes
gc.collect()

print("\nTotal de linhas operacionais selecionadas:", f"{len(operacoes):,}")
print(
    "Combinações REF_BACEN + NU_ORDEM:",
    f"{operacoes[['REF_BACEN', 'NU_ORDEM']].drop_duplicates().shape[0]:,}",
)

display(
    pd.DataFrame(
        [
            {"ANO": ano, "LINHAS_SELECIONADAS": totais_operacoes_ano[ano]}
            for ano in ANOS
        ]
    )
)



PROCESSANDO OPERAÇÕES DE 2021
Bloco 001: 4,448 linhas | acumulado do ano=4,448
Bloco 002: 3,840 linhas | acumulado do ano=8,288
Bloco 003: 2,699 linhas | acumulado do ano=10,987
Bloco 004: 3,261 linhas | acumulado do ano=14,248
Bloco 005: 3,617 linhas | acumulado do ano=17,865
Bloco 006: 3,077 linhas | acumulado do ano=20,942
Bloco 007: 2,037 linhas | acumulado do ano=22,979
Bloco 008: 2,211 linhas | acumulado do ano=25,190
Bloco 009: 3,140 linhas | acumulado do ano=28,330
Bloco 010: 3,548 linhas | acumulado do ano=31,878
Bloco 011: 3,977 linhas | acumulado do ano=35,855
Bloco 012: 4,208 linhas | acumulado do ano=40,063
Bloco 013: 4,673 linhas | acumulado do ano=44,736
Bloco 014: 5,749 linhas | acumulado do ano=50,485
Bloco 015: 5,325 linhas | acumulado do ano=55,810
Bloco 016: 6,926 linhas | acumulado do ano=62,736
Bloco 017: 6,024 linhas | acumulado do ano=68,760
Bloco 018: 7,915 linhas | acumulado do ano=76,675
Bloco 019: 7,230 linhas | acumulado do ano=83,905
Bloco 020: 6,295 linh

,ANO,LINHAS_SELECIONADAS
0,2021,90200
1,2022,76665
2,2023,77546
3,2024,83869
4,2025,73014


In [9]:
# ============================================================
# 7. PREPARAÇÃO DAS OPERAÇÕES E VINCULAÇÃO DOS DOMÍNIOS
# ============================================================

for coluna in ["CD_PROGRAMA", "CD_CATEG_EMITENTE", "CD_SUBPROGRAMA"]:
    if coluna in operacoes.columns:
        operacoes[coluna] = padronizar_codigo(operacoes[coluna])

for coluna in ["VL_PARC_CREDITO", "VL_AREA_INFORMADA", "VL_AREA_FINANC"]:
    if coluna in operacoes.columns:
        operacoes[coluna] = numero_brasileiro(operacoes[coluna])

if "DT_EMISSAO" in operacoes.columns:
    operacoes["DT_EMISSAO"] = pd.to_datetime(
        operacoes["DT_EMISSAO"],
        errors="coerce",
        dayfirst=True,
    )
    operacoes["ANO_OPERACAO"] = (
        operacoes["DT_EMISSAO"].dt.year.astype("Int64")
    )
else:
    operacoes["ANO_OPERACAO"] = (
        pd.to_numeric(
            operacoes["ANO_ARQUIVO_OPERACAO"],
            errors="coerce",
        ).astype("Int64")
    )

MAPA_UF_IBGE = {
    "11": "RO", "12": "AC", "13": "AM", "14": "RR", "15": "PA",
    "16": "AP", "17": "TO", "21": "MA", "22": "PI", "23": "CE",
    "24": "RN", "25": "PB", "26": "PE", "27": "AL", "28": "SE",
    "29": "BA", "31": "MG", "32": "ES", "33": "RJ", "35": "SP",
    "41": "PR", "42": "SC", "43": "RS", "50": "MS", "51": "MT",
    "52": "GO", "53": "DF",
}

if "SG_UF" in operacoes.columns:
    operacoes["UF_OPERACAO"] = (
        operacoes["SG_UF"].astype("string").str.strip().str.upper()
    )
elif "CD_ESTADO" in operacoes.columns:
    operacoes["UF_OPERACAO"] = (
        padronizar_codigo(operacoes["CD_ESTADO"])
        .str.zfill(2)
        .map(MAPA_UF_IBGE)
    )
else:
    operacoes["UF_OPERACAO"] = pd.NA

if "CD_PROGRAMA" in operacoes.columns:
    operacoes = operacoes.merge(
        programa_dim,
        on="CD_PROGRAMA",
        how="left",
        validate="many_to_one",
    )

if "CD_CATEG_EMITENTE" in operacoes.columns:
    operacoes = operacoes.merge(
        categoria_dim,
        on="CD_CATEG_EMITENTE",
        how="left",
        validate="many_to_one",
    )

if "CD_SUBPROGRAMA" in operacoes.columns:
    if "CD_PROGRAMA" in subprograma_dim.columns:
        operacoes = operacoes.merge(
            subprograma_dim,
            on=["CD_PROGRAMA", "CD_SUBPROGRAMA"],
            how="left",
            validate="many_to_one",
        )
    else:
        operacoes = operacoes.merge(
            subprograma_dim,
            on="CD_SUBPROGRAMA",
            how="left",
            validate="many_to_one",
        )

print("Operações preparadas e domínios vinculados.")
display(operacoes.head())


Operações preparadas e domínios vinculados.


,REF_BACEN,NU_ORDEM,DT_EMISSAO,CD_CATEG_EMITENTE,CD_ESTADO,CD_PROGRAMA,VL_PARC_CREDITO,VL_AREA_FINANC,CD_SUBPROGRAMA,VL_AREA_INFORMADA,ANO_ARQUIVO_OPERACAO,ANO_OPERACAO,UF_OPERACAO,DESC_PROGRAMA,DESC_CATEG_EMITENTE,DESC_SUBPROGRAMA
0,511405404,1,2021-01-04,2222,MG,0001,31582.32,3.0,0001,3.0,2021,2021,NaN,PRONAF - PROGRAMA NACIONAL DE FORTALECIMENTO DA AGRICULTURA FAMILIAR,Pequeno Produtor Rural,Custeio (MCR 10-4)
1,511405405,1,2021-01-04,2222,MG,0001,48809.04,3.0,0001,3.0,2021,2021,NaN,PRONAF - PROGRAMA NACIONAL DE FORTALECIMENTO DA AGRICULTURA FAMILIAR,Pequeno Produtor Rural,Custeio (MCR 10-4)
2,511405424,1,2021-01-04,2222,MG,0001,41093.1,30.0,0001,30.0,2021,2021,NaN,PRONAF - PROGRAMA NACIONAL DE FORTALECIMENTO DA AGRICULTURA FAMILIAR,Pequeno Produtor Rural,Custeio (MCR 10-4)
3,511405446,1,2021-01-04,2222,MG,0001,23837.76,71.99,0001,71.99,2021,2021,NaN,PRONAF - PROGRAMA NACIONAL DE FORTALECIMENTO DA AGRICULTURA FAMILIAR,Pequeno Produtor Rural,Custeio (MCR 10-4)
4,511405448,1,2021-01-04,2222,MG,0001,52427.59,20.1,0001,20.1,2021,2021,NaN,PRONAF - PROGRAMA NACIONAL DE FORTALECIMENTO DA AGRICULTURA FAMILIAR,Pequeno Produtor Rural,Custeio (MCR 10-4)


In [10]:
# ============================================================
# 8. CONSOLIDAÇÃO TÉCNICA SOMENTE POR REF_BACEN + NU_ORDEM
# ============================================================

chave_ordem = ["REF_BACEN", "NU_ORDEM"]

campos_ordem = [
    coluna
    for coluna in operacoes.columns
    if coluna not in chave_ordem
]

agregacoes_ordem = {
    coluna: primeiro_nao_nulo
    for coluna in campos_ordem
}

operacoes_por_ordem = (
    operacoes.groupby(
        chave_ordem,
        dropna=False,
        as_index=False,
    )
    .agg(agregacoes_ordem)
)

quantidade_linhas_por_ordem = (
    operacoes.groupby(
        chave_ordem,
        dropna=False,
    )
    .size()
    .rename("QUANTIDADE_LINHAS_OPERACIONAIS_ORIGEM")
    .reset_index()
)

operacoes_por_ordem = operacoes_por_ordem.merge(
    quantidade_linhas_por_ordem,
    on=chave_ordem,
    how="left",
    validate="one_to_one",
)

campos_auditoria = [
    coluna
    for coluna in [
        "DT_EMISSAO",
        "UF_OPERACAO",
        "CD_PROGRAMA",
        "CD_CATEG_EMITENTE",
        "CD_SUBPROGRAMA",
        "VL_PARC_CREDITO",
        "VL_AREA_INFORMADA",
        "VL_AREA_FINANC",
    ]
    if coluna in operacoes.columns
]

registros_auditoria = []

for campo in campos_auditoria:
    quantidades = (
        operacoes.groupby(chave_ordem, dropna=False)[campo]
        .nunique(dropna=True)
    )
    registros_auditoria.append(
        {
            "CAMPO": campo,
            "ORDENS_COM_MAIS_DE_UM_VALOR_DISTINTO": int(
                (quantidades > 1).sum()
            ),
            "REGRA_APLICADA": "Primeiro valor não nulo por REF_BACEN + NU_ORDEM",
        }
    )

auditoria_ordens = pd.DataFrame(registros_auditoria)

# Campos analíticos agora são calculados por ordem, e não por REF_BACEN agregado.
desc_categoria = (
    texto_sem_acento_maiusculo(
        operacoes_por_ordem["DESC_CATEG_EMITENTE"]
    )
    if "DESC_CATEG_EMITENTE" in operacoes_por_ordem.columns
    else pd.Series("", index=operacoes_por_ordem.index)
)

desc_programa = (
    texto_sem_acento_maiusculo(
        operacoes_por_ordem["DESC_PROGRAMA"]
    )
    if "DESC_PROGRAMA" in operacoes_por_ordem.columns
    else pd.Series("", index=operacoes_por_ordem.index)
)

desc_subprograma = (
    texto_sem_acento_maiusculo(
        operacoes_por_ordem["DESC_SUBPROGRAMA"]
    )
    if "DESC_SUBPROGRAMA" in operacoes_por_ordem.columns
    else pd.Series("", index=operacoes_por_ordem.index)
)

codigo_categoria = (
    operacoes_por_ordem["CD_CATEG_EMITENTE"].astype("string")
    if "CD_CATEG_EMITENTE" in operacoes_por_ordem.columns
    else pd.Series(
        pd.NA,
        index=operacoes_por_ordem.index,
        dtype="string",
    )
)

cond_pequeno = (
    codigo_categoria.str.contains("2222", na=False)
    | desc_categoria.str.contains("PEQUEN", na=False)
    | desc_programa.str.contains("PRONAF", na=False)
)

cond_medio = (
    codigo_categoria.str.contains("3333", na=False)
    | desc_categoria.str.contains("MEDIO", na=False)
    | desc_programa.str.contains("PRONAMP", na=False)
)

cond_grande = (
    codigo_categoria.str.contains("4444", na=False)
    | desc_categoria.str.contains("GRANDE", na=False)
)

operacoes_por_ordem["TIPO_PRODUTOR_ANALISE"] = np.select(
    [cond_pequeno, cond_medio, cond_grande],
    [
        "Pequeno produtor / Pronaf",
        "Médio produtor / Pronamp",
        "Grande produtor",
    ],
    default="Demais produtores ou não classificado",
)

operacoes_por_ordem["IND_PRONAF_GRUPO_B"] = np.where(
    desc_subprograma.str.contains(r"GRUPO\s*B", regex=True, na=False),
    "Sim",
    "Não",
)

print("Linhas operacionais antes:", f"{len(operacoes):,}")
print(
    "Ordens únicas após consolidação técnica:",
    f"{len(operacoes_por_ordem):,}",
)
print(
    "Ordens representadas por mais de uma linha operacional:",
    f"{(quantidade_linhas_por_ordem['QUANTIDADE_LINHAS_OPERACIONAIS_ORIGEM'] > 1).sum():,}",
)

display(auditoria_ordens)
display(operacoes_por_ordem.head())


Linhas operacionais antes: 401,294
Ordens únicas após consolidação técnica: 401,294
Ordens representadas por mais de uma linha operacional: 0


,CAMPO,ORDENS_COM_MAIS_DE_UM_VALOR_DISTINTO,REGRA_APLICADA
0,DT_EMISSAO,0,Primeiro valor não nulo por REF_BACEN + NU_ORDEM
1,UF_OPERACAO,0,Primeiro valor não nulo por REF_BACEN + NU_ORDEM
2,CD_PROGRAMA,0,Primeiro valor não nulo por REF_BACEN + NU_ORDEM
3,CD_CATEG_EMITENTE,0,Primeiro valor não nulo por REF_BACEN + NU_ORDEM
4,CD_SUBPROGRAMA,0,Primeiro valor não nulo por REF_BACEN + NU_ORDEM
5,VL_PARC_CREDITO,0,Primeiro valor não nulo por REF_BACEN + NU_ORDEM
6,VL_AREA_INFORMADA,0,Primeiro valor não nulo por REF_BACEN + NU_ORDEM
7,VL_AREA_FINANC,0,Primeiro valor não nulo por REF_BACEN + NU_ORDEM


,REF_BACEN,NU_ORDEM,DT_EMISSAO,CD_CATEG_EMITENTE,CD_ESTADO,CD_PROGRAMA,VL_PARC_CREDITO,VL_AREA_FINANC,CD_SUBPROGRAMA,VL_AREA_INFORMADA,ANO_ARQUIVO_OPERACAO,ANO_OPERACAO,UF_OPERACAO,DESC_PROGRAMA,DESC_CATEG_EMITENTE,DESC_SUBPROGRAMA,QUANTIDADE_LINHAS_OPERACIONAIS_ORIGEM,TIPO_PRODUTOR_ANALISE,IND_PRONAF_GRUPO_B
0,511405404,1,2021-01-04,2222,MG,0001,31582.32,3.0,0001,3.0,2021,2021,<NA>,PRONAF - PROGRAMA NACIONAL DE FORTALECIMENTO DA AGRICULTURA FAMILIAR,Pequeno Produtor Rural,Custeio (MCR 10-4),1,Pequeno produtor / Pronaf,Não
1,511405405,1,2021-01-04,2222,MG,0001,48809.04,3.0,0001,3.0,2021,2021,<NA>,PRONAF - PROGRAMA NACIONAL DE FORTALECIMENTO DA AGRICULTURA FAMILIAR,Pequeno Produtor Rural,Custeio (MCR 10-4),1,Pequeno produtor / Pronaf,Não
2,511405424,1,2021-01-04,2222,MG,0001,41093.1,30.0,0001,30.0,2021,2021,<NA>,PRONAF - PROGRAMA NACIONAL DE FORTALECIMENTO DA AGRICULTURA FAMILIAR,Pequeno Produtor Rural,Custeio (MCR 10-4),1,Pequeno produtor / Pronaf,Não
3,511405446,1,2021-01-04,2222,MG,0001,23837.76,71.99,0001,71.99,2021,2021,<NA>,PRONAF - PROGRAMA NACIONAL DE FORTALECIMENTO DA AGRICULTURA FAMILIAR,Pequeno Produtor Rural,Custeio (MCR 10-4),1,Pequeno produtor / Pronaf,Não
4,511405448,1,2021-01-04,2222,MG,0001,52427.59,20.1,0001,20.1,2021,2021,<NA>,PRONAF - PROGRAMA NACIONAL DE FORTALECIMENTO DA AGRICULTURA FAMILIAR,Pequeno Produtor Rural,Custeio (MCR 10-4),1,Pequeno produtor / Pronaf,Não


In [11]:
# ============================================================
# 9. ENRIQUECIMENTO DA BASE TERRITORIAL DETALHADA
# ============================================================

operacoes_para_merge = operacoes_por_ordem.rename(
    columns={"NU_ORDEM": "NU_ORDEM_OPERACAO"}
)

resultado = glebas.merge(
    operacoes_para_merge,
    left_on=["REF_BACEN", "NU_ORDEM_GLEBA"],
    right_on=["REF_BACEN", "NU_ORDEM_OPERACAO"],
    how="left",
    validate="many_to_one",
    indicator="_VINCULO_OPERACAO",
    sort=False,
)

resultado = (
    resultado.sort_values("_ORDEM_LINHA_BASE04", kind="stable")
    .reset_index(drop=True)
)

resultado["VINCULO_OPERACAO"] = np.where(
    resultado["_VINCULO_OPERACAO"].eq("both"),
    "Vinculado",
    "Não vinculado",
)

resultado = resultado.drop(columns=["_VINCULO_OPERACAO"])

resultado["CHAVE_ORDEM_CREDITO"] = (
    resultado["REF_BACEN"].astype("string")
    + "_"
    + resultado["NU_ORDEM_GLEBA"].astype("string")
)

# A junção many_to_one deve preservar exatamente todas as colunas territoriais.
if len(resultado) != quantidade_registros_origem:
    raise RuntimeError(
        "A junção alterou a quantidade de registros territoriais."
    )

if not resultado["ID_GLEBA"].is_unique:
    raise RuntimeError("O resultado contém ID_GLEBA duplicados.")

if not resultado[colunas_originais_base].equals(
    glebas[colunas_originais_base].reset_index(drop=True)
):
    raise RuntimeError(
        "Uma ou mais colunas territoriais foram alteradas durante a junção."
    )

quantidade_vinculada = int(
    resultado["VINCULO_OPERACAO"].eq("Vinculado").sum()
)
quantidade_nao_vinculada = int(
    resultado["VINCULO_OPERACAO"].eq("Não vinculado").sum()
)
taxa_vinculo = (
    quantidade_vinculada / len(resultado) * 100
    if len(resultado)
    else 0
)

# Organiza o arquivo com os campos territoriais primeiro.
colunas_enriquecidas = [
    coluna
    for coluna in resultado.columns
    if coluna not in colunas_originais_base
    and coluna != "_ORDEM_LINHA_BASE04"
]

resultado = resultado[
    colunas_originais_base + colunas_enriquecidas
]

print("VALIDAÇÃO DA JUNÇÃO")
print("=" * 80)
print("Linhas territoriais na origem:", f"{quantidade_registros_origem:,}")
print("Linhas no resultado:", f"{len(resultado):,}")
print("ID_GLEBA únicos:", f"{resultado['ID_GLEBA'].nunique():,}")
print("REF_BACEN distintos:", f"{resultado['REF_BACEN'].nunique():,}")
print("Registros vinculados:", f"{quantidade_vinculada:,}")
print("Registros não vinculados:", f"{quantidade_nao_vinculada:,}")
print("Taxa de vinculação:", f"{taxa_vinculo:.2f}%")

display(resultado.head())


VALIDAÇÃO DA JUNÇÃO
Linhas territoriais na origem: 480,679
Linhas no resultado: 480,679
ID_GLEBA únicos: 480,679
REF_BACEN distintos: 334,166
Registros vinculados: 480,679
Registros não vinculados: 0
Taxa de vinculação: 100.00%


,ID_GLEBA,REF_BACEN,NU_ORDEM_GLEBA,NU_INDICE,GT_GEOMETRIA,ANO_BASE,NU_ORDEM_OPERACAO,DT_EMISSAO,CD_CATEG_EMITENTE,CD_ESTADO,CD_PROGRAMA,VL_PARC_CREDITO,VL_AREA_FINANC,CD_SUBPROGRAMA,VL_AREA_INFORMADA,ANO_ARQUIVO_OPERACAO,ANO_OPERACAO,UF_OPERACAO,DESC_PROGRAMA,DESC_CATEG_EMITENTE,DESC_SUBPROGRAMA,QUANTIDADE_LINHAS_OPERACIONAIS_ORIGEM,TIPO_PRODUTOR_ANALISE,IND_PRONAF_GRUPO_B,VINCULO_OPERACAO,CHAVE_ORDEM_CREDITO
0,1,511405404,1,0,"POLYGON ((-47.521674 -18.527653 0,-47.521697 -18.528289 0,-47.519128 -18.528105 0,-47.519506 -18.526262 0,-47.519955 -18.526712 0,-47.520342 -18.527134 0,-47.520581 -18.527374 ...",2021,1,2021-01-04,2222,MG,0001,31582.32,3.0,0001,3.0,2021,2021,<NA>,PRONAF - PROGRAMA NACIONAL DE FORTALECIMENTO DA AGRICULTURA FAMILIAR,Pequeno Produtor Rural,Custeio (MCR 10-4),1,Pequeno produtor / Pronaf,Não,Vinculado,511405404_1
1,2,511405405,1,0,"POLYGON ((-47.579356 -18.906067 0,-47.578942 -18.906589 0,-47.578871 -18.907144 0,-47.578482 -18.907021 0,-47.57761 -18.906084 0,-47.577963 -18.905703 0,-47.57713 -18.905449999...",2021,1,2021-01-04,2222,MG,0001,48809.04,3.0,0001,3.0,2021,2021,<NA>,PRONAF - PROGRAMA NACIONAL DE FORTALECIMENTO DA AGRICULTURA FAMILIAR,Pequeno Produtor Rural,Custeio (MCR 10-4),1,Pequeno produtor / Pronaf,Não,Vinculado,511405405_1
2,3,511405424,1,0,"POLYGON ((-44.441404 -16.832015 0,-44.441419 -16.831524 0,-44.440824 -16.831596 0,-44.439808999999897 -16.831699 0,-44.43956 -16.830835 0,-44.439273 -16.830788 0,-44.43887 -16....",2021,1,2021-01-04,2222,MG,0001,41093.1,30.0,0001,30.0,2021,2021,<NA>,PRONAF - PROGRAMA NACIONAL DE FORTALECIMENTO DA AGRICULTURA FAMILIAR,Pequeno Produtor Rural,Custeio (MCR 10-4),1,Pequeno produtor / Pronaf,Não,Vinculado,511405424_1
3,4,511405446,1,0,"POLYGON ((-44.965694 -20.079657 0,-44.967058 -20.077494 0,-44.968458 -20.078419 0,-44.970597 -20.078106 0,-44.972607 -20.077254 0,-44.972477 -20.076066 0,-44.972887 -20.074928 ...",2021,1,2021-01-04,2222,MG,0001,23837.76,71.99,0001,71.99,2021,2021,<NA>,PRONAF - PROGRAMA NACIONAL DE FORTALECIMENTO DA AGRICULTURA FAMILIAR,Pequeno Produtor Rural,Custeio (MCR 10-4),1,Pequeno produtor / Pronaf,Não,Vinculado,511405446_1
4,5,511405448,1,0,"POLYGON ((-45.075154 -18.327891 0,-45.077628 -18.329547 0,-45.078869 -18.329854 0,-45.079659 -18.329899 0,-45.08048 -18.330235 0,-45.081125 -18.331326 0,-45.081536 -18.331539 0...",2021,1,2021-01-04,2222,MG,0001,52427.59,20.1,0001,20.1,2021,2021,<NA>,PRONAF - PROGRAMA NACIONAL DE FORTALECIMENTO DA AGRICULTURA FAMILIAR,Pequeno Produtor Rural,Custeio (MCR 10-4),1,Pequeno produtor / Pronaf,Não,Vinculado,511405448_1


In [12]:
# ============================================================
# 10. RELATÓRIO, DICIONÁRIO E EXPORTAÇÃO
# ============================================================

contagem_geometrias = resultado["GT_GEOMETRIA"].value_counts(dropna=True)
quantidade_geometrias_distintas = int(len(contagem_geometrias))
quantidade_geometrias_repetidas = int((contagem_geometrias > 1).sum())

relatorio = pd.DataFrame(
    [
        {
            "INDICADOR": "Base territorial utilizada",
            "VALOR": str(ARQUIVO_CERRADO_CONSOLIDADO),
        },
        {
            "INDICADOR": "Lista auxiliar de REF_BACEN únicos utilizada",
            "VALOR": "Não",
        },
        {
            "INDICADOR": "Registros territoriais na origem",
            "VALOR": quantidade_registros_origem,
        },
        {
            "INDICADOR": "Registros territoriais no resultado",
            "VALOR": len(resultado),
        },
        {
            "INDICADOR": "ID_GLEBA distintos no resultado",
            "VALOR": resultado["ID_GLEBA"].nunique(dropna=True),
        },
        {
            "INDICADOR": "REF_BACEN distintos na base territorial",
            "VALOR": quantidade_refs_origem,
        },
        {
            "INDICADOR": "Linhas com REF_BACEN repetido além da primeira ocorrência",
            "VALOR": resultado.duplicated(subset=["REF_BACEN"]).sum(),
        },
        {
            "INDICADOR": "Geometrias WKT distintas",
            "VALOR": quantidade_geometrias_distintas,
        },
        {
            "INDICADOR": "Geometrias WKT com mais de uma ocorrência",
            "VALOR": quantidade_geometrias_repetidas,
        },
        {
            "INDICADOR": "Ordens operacionais únicas usadas no enriquecimento",
            "VALOR": len(operacoes_por_ordem),
        },
        {
            "INDICADOR": "Registros territoriais vinculados às operações",
            "VALOR": quantidade_vinculada,
        },
        {
            "INDICADOR": "Registros territoriais não vinculados às operações",
            "VALOR": quantidade_nao_vinculada,
        },
        {
            "INDICADOR": "Taxa de vinculação (%)",
            "VALOR": round(taxa_vinculo, 2),
        },
        {
            "INDICADOR": "Quantidade de colunas no resultado",
            "VALOR": len(resultado.columns),
        },
    ]
)

descricoes = {
    "ID_GLEBA": "Identificador único sequencial de cada registro territorial.",
    "REF_BACEN": "Referência da operação de crédito no SICOR; pode repetir.",
    "NU_ORDEM_GLEBA": "Número da ordem associado ao registro territorial.",
    "NU_INDICE": "Índice da geometria dentro da ordem.",
    "GT_GEOMETRIA": "Geometria WKT preservada integralmente da base territorial.",
    "ANO_BASE": "Ano de origem do registro territorial.",
    "NU_ORDEM_OPERACAO": "Número da ordem localizado na base operacional.",
    "CHAVE_ORDEM_CREDITO": "Combinação REF_BACEN + NU_ORDEM; usar para não somar valores financeiros repetidos entre glebas.",
    "QUANTIDADE_LINHAS_OPERACIONAIS_ORIGEM": "Quantidade de linhas operacionais consolidadas para a mesma ordem.",
    "TIPO_PRODUTOR_ANALISE": "Classificação analítica do produtor calculada por ordem.",
    "IND_PRONAF_GRUPO_B": "Indica se a descrição do subprograma menciona Pronaf Grupo B.",
    "VINCULO_OPERACAO": "Indica se REF_BACEN + NU_ORDEM foi localizado nas bases operacionais.",
}

dicionario = pd.DataFrame(
    {
        "CAMPO": resultado.columns,
        "TIPO": [str(resultado[coluna].dtype) for coluna in resultado.columns],
        "DESCRICAO": [
            descricoes.get(
                coluna,
                "Campo operacional ou financeiro acrescentado pelo Base04.",
            )
            for coluna in resultado.columns
        ],
    }
)

colunas_nao_vinculados = [
    coluna
    for coluna in [
        "ID_GLEBA",
        "REF_BACEN",
        "NU_ORDEM_GLEBA",
        "NU_INDICE",
        "ANO_BASE",
        "GT_GEOMETRIA",
    ]
    if coluna in resultado.columns
]

nao_vinculados = resultado.loc[
    resultado["VINCULO_OPERACAO"].eq("Não vinculado"),
    colunas_nao_vinculados,
].copy()

resultado.to_csv(
    ARQUIVO_SAIDA,
    index=False,
    sep=SEPARADOR_SAIDA,
    decimal=",",
    encoding="utf-8-sig",
)

relatorio.to_csv(
    ARQUIVO_RELATORIO,
    index=False,
    sep=SEPARADOR_SAIDA,
    encoding="utf-8-sig",
)

nao_vinculados.to_csv(
    ARQUIVO_NAO_VINCULADOS,
    index=False,
    sep=SEPARADOR_SAIDA,
    encoding="utf-8-sig",
)

auditoria_ordens.to_csv(
    ARQUIVO_AUDITORIA_ORDENS,
    index=False,
    sep=SEPARADOR_SAIDA,
    encoding="utf-8-sig",
)

dicionario.to_csv(
    ARQUIVO_DICIONARIO,
    index=False,
    sep=SEPARADOR_SAIDA,
    encoding="utf-8-sig",
)

print("Arquivos gerados:")
for caminho in [
    ARQUIVO_SAIDA,
    ARQUIVO_RELATORIO,
    ARQUIVO_NAO_VINCULADOS,
    ARQUIVO_AUDITORIA_ORDENS,
    ARQUIVO_DICIONARIO,
]:
    print(f"- {caminho}")

display(relatorio)


Arquivos gerados:
- /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/Base04_enriquecimento_detalhado/sicor_cerrado_2021_2025_enriquecido_detalhado.csv
- /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/Base04_enriquecimento_detalhado/relatorio_base04_enriquecimento_detalhado.csv
- /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/Base04_enriquecimento_detalhado/registros_territoriais_nao_vinculados_operacoes.csv
- /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/Base04_enriquecimento_detalhado/auditoria_consolidacao_operacoes_por_ordem.csv
- /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/Base04_enriquecimento_detalhado/dicionario_base04_enriquecimento_detalhado.csv


,INDICADOR,VALOR
0,Base territorial utilizada,/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/sicor_cerrado_2021_2025.csv
1,Lista auxiliar de REF_BACEN únicos utilizada,Não
2,Registros territoriais na origem,480679
3,Registros territoriais no resultado,480679
4,ID_GLEBA distintos no resultado,480679
5,REF_BACEN distintos na base territorial,334166
6,Linhas com REF_BACEN repetido além da primeira ocorrência,146513
7,Geometrias WKT distintas,395575
8,Geometrias WKT com mais de uma ocorrência,57637
9,Ordens operacionais únicas usadas no enriquecimento,401294


In [13]:
# ============================================================
# 11. RELER E VALIDAR O CSV PRINCIPAL EM BLOCOS
# ============================================================

colunas_validacao = [
    "ID_GLEBA",
    "REF_BACEN",
    "NU_ORDEM_GLEBA",
    "NU_INDICE",
    "GT_GEOMETRIA",
    "ANO_BASE",
    "VINCULO_OPERACAO",
]

cabecalho_validacao = pd.read_csv(
    ARQUIVO_SAIDA,
    sep=SEPARADOR_SAIDA,
    encoding="utf-8-sig",
    nrows=0,
)

faltantes_validacao = set(colunas_validacao) - set(cabecalho_validacao.columns)

if faltantes_validacao:
    raise RuntimeError(
        f"O CSV gravado não contém as colunas: {sorted(faltantes_validacao)}"
    )

ids_encontrados = set()
total_validado = 0
duplicados_id = 0
geometrias_vazias = 0
vinculados_validados = 0

leitor_validacao = pd.read_csv(
    ARQUIVO_SAIDA,
    sep=SEPARADOR_SAIDA,
    encoding="utf-8-sig",
    usecols=colunas_validacao,
    dtype=str,
    chunksize=TAMANHO_BLOCO,
    low_memory=False,
)

for numero_bloco, bloco in enumerate(leitor_validacao, start=1):
    ids_bloco = bloco["ID_GLEBA"].dropna().astype(str)

    duplicados_id += int(ids_bloco.duplicated().sum())
    duplicados_id += sum(
        identificador in ids_encontrados
        for identificador in ids_bloco.unique()
    )
    ids_encontrados.update(ids_bloco.unique())

    geometrias_vazias += int(
        bloco["GT_GEOMETRIA"].isna().sum()
        + bloco["GT_GEOMETRIA"].fillna("").str.strip().eq("").sum()
    )
    vinculados_validados += int(
        bloco["VINCULO_OPERACAO"].eq("Vinculado").sum()
    )
    total_validado += len(bloco)

    print(
        f"Bloco {numero_bloco:03d}: {len(bloco):,} linhas | "
        f"acumulado={total_validado:,}"
    )

    del bloco, ids_bloco
    gc.collect()

if total_validado != quantidade_registros_origem:
    raise RuntimeError(
        "O CSV gravado não preservou a quantidade de registros territoriais."
    )

if duplicados_id != 0:
    raise RuntimeError(
        f"O CSV gravado contém {duplicados_id:,} ID_GLEBA duplicados."
    )

if geometrias_vazias != 0:
    raise RuntimeError(
        f"O CSV gravado contém {geometrias_vazias:,} geometrias vazias."
    )

if vinculados_validados != quantidade_vinculada:
    raise RuntimeError(
        "A quantidade de vínculos mudou após a gravação do CSV."
    )

print("\nVALIDAÇÃO DO ARQUIVO GRAVADO APROVADA")
print("=" * 80)
print("Linhas:", f"{total_validado:,}")
print("ID_GLEBA distintos:", f"{len(ids_encontrados):,}")
print("Geometrias vazias:", f"{geometrias_vazias:,}")
print("Registros vinculados:", f"{vinculados_validados:,}")
print("Colunas:", f"{len(cabecalho_validacao.columns):,}")

amostra_validacao = pd.read_csv(
    ARQUIVO_SAIDA,
    sep=SEPARADOR_SAIDA,
    encoding="utf-8-sig",
    nrows=5,
    low_memory=False,
)

display(amostra_validacao)


Bloco 001: 100,000 linhas | acumulado=100,000
Bloco 002: 100,000 linhas | acumulado=200,000
Bloco 003: 100,000 linhas | acumulado=300,000
Bloco 004: 100,000 linhas | acumulado=400,000
Bloco 005: 80,679 linhas | acumulado=480,679

VALIDAÇÃO DO ARQUIVO GRAVADO APROVADA
Linhas: 480,679
ID_GLEBA distintos: 480,679
Geometrias vazias: 0
Registros vinculados: 480,679
Colunas: 26


,ID_GLEBA,REF_BACEN,NU_ORDEM_GLEBA,NU_INDICE,GT_GEOMETRIA,ANO_BASE,NU_ORDEM_OPERACAO,DT_EMISSAO,CD_CATEG_EMITENTE,CD_ESTADO,CD_PROGRAMA,VL_PARC_CREDITO,VL_AREA_FINANC,CD_SUBPROGRAMA,VL_AREA_INFORMADA,ANO_ARQUIVO_OPERACAO,ANO_OPERACAO,UF_OPERACAO,DESC_PROGRAMA,DESC_CATEG_EMITENTE,DESC_SUBPROGRAMA,QUANTIDADE_LINHAS_OPERACIONAIS_ORIGEM,TIPO_PRODUTOR_ANALISE,IND_PRONAF_GRUPO_B,VINCULO_OPERACAO,CHAVE_ORDEM_CREDITO
0,1,511405404,1,0,"POLYGON ((-47.521674 -18.527653 0,-47.521697 -18.528289 0,-47.519128 -18.528105 0,-47.519506 -18.526262 0,-47.519955 -18.526712 0,-47.520342 -18.527134 0,-47.520581 -18.527374 ...",2021,1,2021-01-04,2222,MG,1,"31582,32","3,0",1,"3,0",2021,2021,NaN,PRONAF - PROGRAMA NACIONAL DE FORTALECIMENTO DA AGRICULTURA FAMILIAR,Pequeno Produtor Rural,Custeio (MCR 10-4),1,Pequeno produtor / Pronaf,Não,Vinculado,511405404_1
1,2,511405405,1,0,"POLYGON ((-47.579356 -18.906067 0,-47.578942 -18.906589 0,-47.578871 -18.907144 0,-47.578482 -18.907021 0,-47.57761 -18.906084 0,-47.577963 -18.905703 0,-47.57713 -18.905449999...",2021,1,2021-01-04,2222,MG,1,"48809,04","3,0",1,"3,0",2021,2021,NaN,PRONAF - PROGRAMA NACIONAL DE FORTALECIMENTO DA AGRICULTURA FAMILIAR,Pequeno Produtor Rural,Custeio (MCR 10-4),1,Pequeno produtor / Pronaf,Não,Vinculado,511405405_1
2,3,511405424,1,0,"POLYGON ((-44.441404 -16.832015 0,-44.441419 -16.831524 0,-44.440824 -16.831596 0,-44.439808999999897 -16.831699 0,-44.43956 -16.830835 0,-44.439273 -16.830788 0,-44.43887 -16....",2021,1,2021-01-04,2222,MG,1,"41093,1","30,0",1,"30,0",2021,2021,NaN,PRONAF - PROGRAMA NACIONAL DE FORTALECIMENTO DA AGRICULTURA FAMILIAR,Pequeno Produtor Rural,Custeio (MCR 10-4),1,Pequeno produtor / Pronaf,Não,Vinculado,511405424_1
3,4,511405446,1,0,"POLYGON ((-44.965694 -20.079657 0,-44.967058 -20.077494 0,-44.968458 -20.078419 0,-44.970597 -20.078106 0,-44.972607 -20.077254 0,-44.972477 -20.076066 0,-44.972887 -20.074928 ...",2021,1,2021-01-04,2222,MG,1,"23837,76","71,99",1,"71,99",2021,2021,NaN,PRONAF - PROGRAMA NACIONAL DE FORTALECIMENTO DA AGRICULTURA FAMILIAR,Pequeno Produtor Rural,Custeio (MCR 10-4),1,Pequeno produtor / Pronaf,Não,Vinculado,511405446_1
4,5,511405448,1,0,"POLYGON ((-45.075154 -18.327891 0,-45.077628 -18.329547 0,-45.078869 -18.329854 0,-45.079659 -18.329899 0,-45.08048 -18.330235 0,-45.081125 -18.331326 0,-45.081536 -18.331539 0...",2021,1,2021-01-04,2222,MG,1,"52427,59","20,1",1,"20,1",2021,2021,NaN,PRONAF - PROGRAMA NACIONAL DE FORTALECIMENTO DA AGRICULTURA FAMILIAR,Pequeno Produtor Rural,Custeio (MCR 10-4),1,Pequeno produtor / Pronaf,Não,Vinculado,511405448_1


## Produto principal

O arquivo enriquecido detalhado será salvo em:

```text
/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/
Arquivos gerados e relatorios/Base04_enriquecimento_detalhado/
sicor_cerrado_2021_2025_enriquecido_detalhado.csv
```

Ele preserva todas as linhas da base territorial. Portanto, os campos financeiros de uma mesma ordem podem aparecer em várias linhas quando a ordem estiver associada a várias geometrias ou índices. Para somar valores financeiros sem duplicação, utilize `CHAVE_ORDEM_CREDITO` uma única vez.

Este arquivo pode ser usado como entrada para as interseções com Terras Indígenas, territórios quilombolas, florestas públicas, assentamentos, unidades de conservação, embargos e demais impedimentos territoriais.
